In [6]:
!pip install --upgrade pydantic>=2.0 openai anthropic google-generativeai together pandas numpy

In [9]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [12]:
# ================================================================
# Install required packages (run this first in Deepnote)
# ================================================================
# !pip install openai anthropic google-generativeai together pandas numpy

# ================================================================
# 2_LLM_INFERENCE_COUNTRY.ipynb
# Predict belief-about-others (%) for Stage 1/2/3/4/5/6/7/8 prompts
# Also computes PI_pred using ground-truth mean_own_willingness (if present)
# ================================================================

import os, re, time, json
import pandas as pd
import numpy as np
from openai import OpenAI
import anthropic
import google.generativeai as genai

# ================================================================
# Keep-Alive Thread (prevents Deepnote timeout)
# ================================================================
import threading

def keep_alive():
    """Print a dot every 60 seconds to show activity"""
    while True:
        print(".", end="", flush=True)
        time.sleep(60)

# Start keep-alive thread
keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("✓ Keep-alive thread started (prevents timeout)\n")


# ================================================================
# Configuration
# ================================================================

SRC = "country_llm_prompts_outcome2_8stages.csv"  # Fixed filename to match script 1
df = pd.read_csv(SRC)
print(f"✅ Loaded {SRC}: {len(df)} countries")

# API keys (set these as environment variables)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  # Google AI Studio API key (for Gemini)
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")  # Together AI (for Llama)

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",  # Google AI Studio: Gemini 2.5 Flash
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"  # Together AI: Meta's Llama 4 Maverick
}

# System instruction to prevent data leakage and ensure lower-bound performance test
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors (including but not limited to André et al., Sparkman et al., Leviston et al., Lees et al., or any other researchers)
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# ================================================================
# Helpers
# ================================================================

def contains_forbidden_strings(text):
    """Check if response contains forbidden strings that suggest academic citation."""
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre",  # Author name variations
        "et al", "et. al", "et.al",  # Citation markers
        "doi", "http://", "https://",  # Links/DOIs
        "paper", "study", "research",  # Academic references
        "published", "journal", "article"  # Publication terms
    ]
    
    return any(term in text_lower for term in forbidden)

def extract_number_0_100(text):
    """Extract a number between 0-100 from text or JSON."""
    if not isinstance(text, str):
        return None
    
    # Try parsing as JSON first
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            # Look for common keys
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    # Fallback: regex extraction
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

# ================================================================
# LLM API callers with constraints
# ================================================================

def call_gpt(prompt, model="gpt-4o", max_retries=3):
    """Call OpenAI API with tools disabled and JSON mode."""
    if not OPENAI_API_KEY:
        print("⚠️  No OpenAI API key found")
        return None
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,  # Deterministic output
                response_format={"type": "json_object"},  # JSON-only output
                # tools parameter omitted = tools disabled by default
                # No tool_choice needed since no tools provided
            )
            content = response.choices[0].message.content
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                print(f"⚠️  GPT attempt {attempt+1}: Forbidden strings detected, retrying...")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    print(f"❌ GPT: All retries exhausted, forbidden strings still present")
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ GPT attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=3):
    """Call Anthropic API with tools disabled."""
    if not CLAUDE_API_KEY:
        print("⚠️  No Claude API key found")
        return None
    
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,  # Deterministic output
                system=SYSTEM_INSTRUCTION,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                # tools=[] means no tools available
                tools=[],
            )
            content = response.content[0].text
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                print(f"⚠️  Claude attempt {attempt+1}: Forbidden strings detected, retrying...")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    print(f"❌ Claude: All retries exhausted, forbidden strings still present")
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Claude attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-2.5-flash", max_retries=3):
    """Call Gemini via Google AI Studio API (using GEMINI_API_KEY)."""
    if not GEMINI_API_KEY:
        print("⚠️  No Google API key found")
        return None
    
    genai.configure(api_key=GEMINI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            model_instance = genai.GenerativeModel(
                model_name=model,
                generation_config={
                    "temperature": 0,  # Deterministic output
                },
            )
            
            full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
            response = model_instance.generate_content(full_prompt)
            content = response.text
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                print(f"⚠️  Gemini attempt {attempt+1}: Forbidden strings detected, retrying...")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    print(f"❌ Gemini: All retries exhausted, forbidden strings still present")
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Gemini attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", max_retries=3):
    """Call Llama via Together AI API."""
    if not LLAMA_API_KEY:
        print("⚠️  No Llama API key found")
        return None
    
    try:
        from together import Together
        client = Together(api_key=LLAMA_API_KEY)
        
        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_INSTRUCTION},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0,
                    max_tokens=100,
                )
                content = response.choices[0].message.content
                
                # Post-validation: check for forbidden strings
                if contains_forbidden_strings(content):
                    print(f"⚠️  Llama attempt {attempt+1}: Forbidden strings detected, retrying...")
                    if attempt < max_retries - 1:
                        time.sleep(2 ** attempt)
                        continue
                    else:
                        print(f"❌ Llama: All retries exhausted, forbidden strings still present")
                        return None
                
                return extract_number_0_100(content)
            except Exception as e:
                print(f"❌ Llama attempt {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
        return None
    except ImportError:
        print("⚠️  Together SDK not installed. Run: pip install together")
        return None

# Map model names to functions
MODEL_CALLERS = {
    "gpt": call_gpt,
    "claude": call_claude,
    "gemini": call_gemini,
    "llama": call_llama,
}

# ================================================================
# Main prediction loop
# ================================================================

all_stages = range(1, 9)
results = []

for STAGE in all_stages:
    print(f"\n{'='*60}")
    print(f"Processing Stage {STAGE}")
    print('='*60)
    
    PROMPT_COL = f"prompt_stage{STAGE}"
    
    if PROMPT_COL not in df.columns:
        print(f"⚠️  Column {PROMPT_COL} not found, skipping Stage {STAGE}")
        continue
    
    stage_results = []
    
    for idx, row in df.iterrows():
        country = row["countrynew"]
        prompt = row[PROMPT_COL]
        
        if pd.isna(prompt):
            print(f"⚠️  No prompt for {country} at Stage {STAGE}, skipping")
            continue
        
        print(f"\nProcessing {country} (Stage {STAGE})...")
        
        row_data = {
            "countrynew": country,
            "stage": STAGE,
        }
        
        # Call each model
        for model_name, caller_func in MODEL_CALLERS.items():
            if model_name not in MODELS:
                continue
                
            print(f"  Calling {model_name}...", end=" ")
            prediction = caller_func(prompt, model=MODELS[model_name])
            
            if prediction is not None:
                print(f"✓ {prediction:.1f}")
                row_data[f"pred_{model_name}"] = prediction
            else:
                print("✗ Failed")
                row_data[f"pred_{model_name}"] = np.nan
            
            # Rate limiting: small delay between API calls
            time.sleep(0.5)
        
        stage_results.append(row_data)
    
    # Convert to DataFrame
    stage_df = pd.DataFrame(stage_results)
    results.append(stage_df)
    
    # Save stage-specific predictions
    stage_tag = f"S{STAGE}_country"
    stage_df.to_csv(f"predictions_{stage_tag}.csv", index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved predictions_{stage_tag}.csv ({len(stage_df)} countries)")

# ================================================================
# Combine all stages
# ================================================================

print(f"\n{'='*60}")
print("Combining all stages...")
print('='*60)

final_pred = pd.concat(results, ignore_index=True)

# Save long format (all predictions)
final_pred.to_csv("predictions_all_stages_long.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved predictions_all_stages_long.csv")

# ================================================================
# Create wide format and compute PI predictions
# ================================================================

# Pivot to wide format: one row per (country, stage)
wide = final_pred.pivot_table(
    index=["countrynew", "stage"],
    values=[col for col in final_pred.columns if col.startswith("pred_")],
    aggfunc="first"  # Should only be one value per country-stage-model
).reset_index()

# Merge with original data
final = df.merge(wide, on="countrynew", how="left")

# Compute PI predictions if ground truth available
pred_cols = [col for col in final.columns if col.startswith("pred_")]
model_names = [col.replace("pred_", "") for col in pred_cols]

if "mean_own_willingness" in final.columns:
    # Convert own willingness to 0-1 scale
    own_share = final["mean_own_willingness"].apply(
        lambda x: (x / 100.0) if (pd.notna(x) and 0 <= x <= 100) else 
                  (x if pd.notna(x) and 0 <= x <= 1 else np.nan)
    )
    
    # Compute PI for each model: PI = own_willingness - predicted_others_willingness
    for pred_col in pred_cols:
        model_name = pred_col.replace("pred_", "")
        pi_col = f"pi_pred_{model_name}"
        final[pi_col] = own_share - (final[pred_col] / 100.0)
    
    # Compute ensemble prediction (mean across models)
    if len(pred_cols) > 0:
        final["pred_ensemble"] = final[pred_cols].mean(axis=1)
        final["pi_pred_ensemble"] = own_share - (final["pred_ensemble"] / 100.0)
        print(f"✅ Computed ensemble predictions across {len(pred_cols)} models")

# ================================================================
# Save final outputs
# ================================================================

# Select relevant columns for wide output
output_cols = ["countrynew", "stage"]

# Add ground truth if available
for col in ["mean_own_willingness", "mean_other_willingness", "pi_gap_true"]:
    if col in final.columns:
        output_cols.append(col)

# Add all prediction columns
output_cols.extend([col for col in final.columns if col.startswith("pred_") or col.startswith("pi_pred_")])

# Filter to existing columns
output_cols = [col for col in output_cols if col in final.columns]

final[output_cols].to_csv("predictions_all_stages_wide.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved predictions_all_stages_wide.csv")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total predictions: {len(final_pred)}")
print(f"Countries: {df['countrynew'].nunique()}")
print(f"Stages: {len(all_stages)}")
print(f"Models: {len([m for m in MODEL_CALLERS.keys() if m in MODELS])}")
print(f"Output files:")
print(f"  - predictions_all_stages_long.csv")
print(f"  - predictions_all_stages_wide.csv")
print(f"  - predictions_S[1-8]_country.csv (per stage)")

# Display sample results
print("\nSample results:")
print(final[output_cols].head(10))

Processing Serbia (Stage 7)...
  Calling gpt... ✓ 35.0
  Calling claude... ✓ 32.5
  Calling gemini... .✓ 15.0
  Calling llama... ✓ 34.5

Processing Sierra Leone (Stage 7)...
  Calling gpt... ✓ 25.0
  Calling claude... ✓ 37.6
  Calling gemini... .✓ 9.5
  Calling llama... ✓ 25.9

Processing Singapore (Stage 7)...
  Calling gpt... ✓ 45.0
  Calling claude... ✓ 37.5
  Calling gemini... ✓ 24.1
  Calling llama... ✓ 34.5

Processing Slovakia (Stage 7)...
  Calling gpt... ✓ 35.0
  Calling claude... ✓ 32.7
  Calling gemini... .✓ 22.5
  Calling llama... ✓ 34.4

Processing Slovenia (Stage 7)...
  Calling gpt... ✓ 35.0
  Calling claude... ✓ 32.5
  Calling gemini... .✓ 28.5
  Calling llama... ✓ 34.5

Processing South Africa (Stage 7)...
  Calling gpt... ✓ 45.0
  Calling claude... ✓ 32.5
  Calling gemini... ✓ 28.5
  Calling llama... ✓ 34.5

Processing South Korea (Stage 7)...
  Calling gpt... ✓ 45.0
  Calling claude... ✓ 37.5
  Calling gemini... .✓ 25.0
  Calling llama... ✓ 24.5

Processing Spain (St

In [10]:
# ================================================================
# Install required packages (run this first in Deepnote)
# ================================================================
# !pip install openai anthropic google-generativeai together pandas numpy

# ================================================================
# 2_LLM_INFERENCE_COUNTRY.ipynb
# Predict belief-about-others (%) for Stage 1/2/3/4/5/6/7/8 prompts
# Also computes PI_pred using ground-truth mean_own_willingness (if present)
# ================================================================

import os, re, time, json
import pandas as pd
import numpy as np
from openai import OpenAI
import anthropic
import google.generativeai as genai

# ================================================================
# Keep-Alive Thread (prevents Deepnote timeout)
# ================================================================
import threading

def keep_alive():
    """Print a dot every 60 seconds to show activity"""
    while True:
        print(".", end="", flush=True)
        time.sleep(60)

# Start keep-alive thread
keep_alive_thread = threading.Thread(target=keep_alive, daemon=True)
keep_alive_thread.start()
print("✓ Keep-alive thread started (prevents timeout)\n")

# ================================================================
# Configuration
# ================================================================

SRC = "country_llm_prompts_outcome2_8stages.csv"  # Fixed filename to match script 1
df = pd.read_csv(SRC)
print(f"✅ Loaded {SRC}: {len(df)} countries")

# API keys (set these as environment variables)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")  # Google AI Studio API key (for Gemini)
LLAMA_API_KEY = os.getenv("LLAMA_API_KEY")  # Together AI (for Llama)

# Model configurations
MODELS = {
    "gpt": "gpt-4o-mini",
    "claude": "claude-3-5-haiku-20241022",
    "gemini": "gemini-2.5-flash",  # Google AI Studio: Gemini 2.5 Flash
    "llama": "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8"  # Together AI: Meta's Llama 4 Maverick
}

# System instruction to prevent data leakage and ensure lower-bound performance test
SYSTEM_INSTRUCTION = """You are a prediction assistant making estimates based ONLY on the information provided in this specific prompt.

CRITICAL INSTRUCTIONS:
1. Do NOT cite, reference, or mention ANY research papers, academic studies, surveys, or authors (including but not limited to André et al., Sparkman et al., Leviston et al., Lees et al., or any other researchers)
2. Do NOT use any memorized data, statistics, or percentages from your training about climate change opinions, pluralistic ignorance, or survey results
3. Treat this as a completely NOVEL scenario - ignore any similar studies you may have seen during training
4. Do NOT reference 'research shows', 'studies indicate', 'surveys have found', or similar phrases
5. Base your estimate ONLY on:
   - General reasoning about human psychology and behavior
   - The specific information provided in this prompt
   - First principles about how people form beliefs about others

Your task is to predict what percentage people THINK others believe (second-order belief), not what people actually believe (first-order belief). This is a prediction task requiring general reasoning, not recall of specific research findings.

Respond with ONLY a JSON object containing a single number between 0 and 100 with one decimal place: {"prediction": XX.X}

Do not include any explanation, reasoning, or text - only the JSON."""

# ================================================================
# Helpers
# ================================================================

def contains_forbidden_strings(text):
    """Check if response contains forbidden strings that suggest academic citation."""
    if not isinstance(text, str):
        return False
    
    text_lower = text.lower()
    forbidden = [
        "andré", "andre",  # Author name variations
        "et al", "et. al", "et.al",  # Citation markers
        "doi", "http://", "https://",  # Links/DOIs
        "paper", "study", "research",  # Academic references
        "published", "journal", "article"  # Publication terms
    ]
    
    return any(term in text_lower for term in forbidden)

def extract_number_0_100(text):
    """Extract a number between 0-100 from text or JSON."""
    if not isinstance(text, str):
        return None
    
    # Try parsing as JSON first
    try:
        data = json.loads(text)
        if isinstance(data, dict):
            # Look for common keys
            for key in ["prediction", "estimate", "value", "number", "percentage"]:
                if key in data:
                    val = float(data[key])
                    return max(0.0, min(100.0, val))
        elif isinstance(data, (int, float)):
            val = float(data)
            return max(0.0, min(100.0, val))
    except:
        pass
    
    # Fallback: regex extraction
    m = re.search(r"(\d+(?:\.\d+)?)", text)
    if not m:
        return None
    val = float(m.group(1))
    return max(0.0, min(100.0, val))

# ================================================================
# LLM API callers with constraints
# ================================================================

def call_gpt(prompt, model="gpt-4o", max_retries=3):
    """Call OpenAI API with tools disabled and JSON mode."""
    if not OPENAI_API_KEY:
        print("⚠️  No OpenAI API key found")
        return None
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_INSTRUCTION},
                    {"role": "user", "content": prompt}
                ],
                temperature=0,  # Deterministic output
                response_format={"type": "json_object"},  # JSON-only output
                # tools parameter omitted = tools disabled by default
                # No tool_choice needed since no tools provided
            )
            content = response.choices[0].message.content
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                print(f"⚠️  GPT attempt {attempt+1}: Forbidden strings detected, retrying...")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    print(f"❌ GPT: All retries exhausted, forbidden strings still present")
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ GPT attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
    return None

def call_claude(prompt, model="claude-3-5-haiku-20241022", max_retries=3):
    """Call Anthropic API with tools disabled."""
    if not CLAUDE_API_KEY:
        print("⚠️  No Claude API key found")
        return None
    
    client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    
    for attempt in range(max_retries):
        try:
            response = client.messages.create(
                model=model,
                max_tokens=100,
                temperature=0,  # Deterministic output
                system=SYSTEM_INSTRUCTION,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                # tools=[] means no tools available
                tools=[],
            )
            content = response.content[0].text
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                print(f"⚠️  Claude attempt {attempt+1}: Forbidden strings detected, retrying...")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    print(f"❌ Claude: All retries exhausted, forbidden strings still present")
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Claude attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_gemini(prompt, model="gemini-2.5-flash", max_retries=3):
    """Call Gemini via Google AI Studio API (using GEMINI_API_KEY)."""
    if not GEMINI_API_KEY:
        print("⚠️  No Google API key found")
        return None
    
    genai.configure(api_key=GEMINI_API_KEY)
    
    for attempt in range(max_retries):
        try:
            model_instance = genai.GenerativeModel(
                model_name=model,
                generation_config={
                    "temperature": 0,  # Deterministic output
                },
            )
            
            full_prompt = f"{SYSTEM_INSTRUCTION}\n\n{prompt}"
            response = model_instance.generate_content(full_prompt)
            content = response.text
            
            # Post-validation: check for forbidden strings
            if contains_forbidden_strings(content):
                print(f"⚠️  Gemini attempt {attempt+1}: Forbidden strings detected, retrying...")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                    continue
                else:
                    print(f"❌ Gemini: All retries exhausted, forbidden strings still present")
                    return None
            
            return extract_number_0_100(content)
        except Exception as e:
            print(f"❌ Gemini attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return None

def call_llama(prompt, model="meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8", max_retries=3):
    """Call Llama via Together AI API."""
    if not LLAMA_API_KEY:
        print("⚠️  No Llama API key found")
        return None
    
    try:
        from together import Together
        client = Together(api_key=LLAMA_API_KEY)
        
        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model=model,
                    messages=[
                        {"role": "system", "content": SYSTEM_INSTRUCTION},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0,
                    max_tokens=100,
                )
                content = response.choices[0].message.content
                
                # Post-validation: check for forbidden strings
                if contains_forbidden_strings(content):
                    print(f"⚠️  Llama attempt {attempt+1}: Forbidden strings detected, retrying...")
                    if attempt < max_retries - 1:
                        time.sleep(2 ** attempt)
                        continue
                    else:
                        print(f"❌ Llama: All retries exhausted, forbidden strings still present")
                        return None
                
                return extract_number_0_100(content)
            except Exception as e:
                print(f"❌ Llama attempt {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
        return None
    except ImportError:
        print("⚠️  Together SDK not installed. Run: pip install together")
        return None

# Map model names to functions
MODEL_CALLERS = {
    "gpt": call_gpt,
    "claude": call_claude,
    "gemini": call_gemini,
    "llama": call_llama,
}

# ================================================================
# Main prediction loop
# ================================================================

# ================================================================
# Main prediction loop
# ================================================================

all_stages = range(1, 9)
results = []

# Check which stages are already completed
import os
completed_stages = []
for s in all_stages:
    if os.path.exists(f"predictions_S{s}_country.csv"):
        completed_stages.append(s)
        print(f"✓ Stage {s} already completed (found predictions_S{s}_country.csv)")

# Ask user if they want to skip completed stages
if completed_stages:
    print(f"\n⚠️  Found {len(completed_stages)} completed stage(s): {completed_stages}")
    print("Options:")
    print("  1. Resume from last incomplete stage (recommended)")
    print("  2. Re-run all stages from scratch")
    
    # For automatic resume, uncomment the next line:
    # choice = "1"
    
    # For manual choice, use this:
    choice = input("Enter choice (1 or 2): ").strip()
    
    if choice == "1":
        # Load completed stages
        for s in completed_stages:
            stage_df = pd.read_csv(f"predictions_S{s}_country.csv")
            results.append(stage_df)
            print(f"  Loaded Stage {s} from file")
        
        # Only process incomplete stages
        remaining_stages = [s for s in all_stages if s not in completed_stages]
        print(f"\n✅ Resuming from Stage {min(remaining_stages) if remaining_stages else 'DONE'}")
        all_stages = remaining_stages
    else:
        print("\n♻️  Re-running all stages from scratch...")

for STAGE in all_stages:
    print(f"\n{'='*60}")
    print(f"Processing Stage {STAGE}")
    print('='*60)
    
    PROMPT_COL = f"prompt_stage{STAGE}"
    
    if PROMPT_COL not in df.columns:
        print(f"⚠️  Column {PROMPT_COL} not found, skipping Stage {STAGE}")
        continue
    
    stage_results = []
    
    for idx, row in df.iterrows():
        country = row["countrynew"]
        prompt = row[PROMPT_COL]
        
        if pd.isna(prompt):
            print(f"⚠️  No prompt for {country} at Stage {STAGE}, skipping")
            continue
        
        print(f"\nProcessing {country} (Stage {STAGE})...")
        
        row_data = {
            "countrynew": country,
            "stage": STAGE,
        }
        
        # Call each model
        for model_name, caller_func in MODEL_CALLERS.items():
            if model_name not in MODELS:
                continue
                
            print(f"  Calling {model_name}...", end=" ")
            prediction = caller_func(prompt, model=MODELS[model_name])
            
            if prediction is not None:
                print(f"✓ {prediction:.1f}")
                row_data[f"pred_{model_name}"] = prediction
            else:
                print("✗ Failed")
                row_data[f"pred_{model_name}"] = np.nan
            
            # Rate limiting: small delay between API calls
            time.sleep(0.5)
        
        stage_results.append(row_data)
    
    # Convert to DataFrame
    stage_df = pd.DataFrame(stage_results)
    results.append(stage_df)
    
    # Save stage-specific predictions
    stage_tag = f"S{STAGE}_country"
    stage_df.to_csv(f"predictions_{stage_tag}.csv", index=False, encoding="utf-8-sig")
    print(f"\n✅ Saved predictions_{stage_tag}.csv ({len(stage_df)} countries)")

# ================================================================
# Combine all stages
# ================================================================

print(f"\n{'='*60}")
print("Combining all stages...")
print('='*60)

final_pred = pd.concat(results, ignore_index=True)

# Save long format (all predictions)
final_pred.to_csv("predictions_all_stages_long.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved predictions_all_stages_long.csv")

# ================================================================
# Create wide format and compute PI predictions
# ================================================================

# Pivot to wide format: one row per (country, stage)
wide = final_pred.pivot_table(
    index=["countrynew", "stage"],
    values=[col for col in final_pred.columns if col.startswith("pred_")],
    aggfunc="first"  # Should only be one value per country-stage-model
).reset_index()

# Merge with original data
final = df.merge(wide, on="countrynew", how="left")

# Compute PI predictions if ground truth available
pred_cols = [col for col in final.columns if col.startswith("pred_")]
model_names = [col.replace("pred_", "") for col in pred_cols]

if "mean_own_willingness" in final.columns:
    # Convert own willingness to 0-1 scale
    own_share = final["mean_own_willingness"].apply(
        lambda x: (x / 100.0) if (pd.notna(x) and 0 <= x <= 100) else 
                  (x if pd.notna(x) and 0 <= x <= 1 else np.nan)
    )
    
    # Compute PI for each model: PI = own_willingness - predicted_others_willingness
    for pred_col in pred_cols:
        model_name = pred_col.replace("pred_", "")
        pi_col = f"pi_pred_{model_name}"
        final[pi_col] = own_share - (final[pred_col] / 100.0)
    
    # Compute ensemble prediction (mean across models)
    if len(pred_cols) > 0:
        final["pred_ensemble"] = final[pred_cols].mean(axis=1)
        final["pi_pred_ensemble"] = own_share - (final["pred_ensemble"] / 100.0)
        print(f"✅ Computed ensemble predictions across {len(pred_cols)} models")

# ================================================================
# Save final outputs
# ================================================================

# Select relevant columns for wide output
output_cols = ["countrynew", "stage"]

# Add ground truth if available
for col in ["mean_own_willingness", "mean_other_willingness", "pi_gap_true"]:
    if col in final.columns:
        output_cols.append(col)

# Add all prediction columns
output_cols.extend([col for col in final.columns if col.startswith("pred_") or col.startswith("pi_pred_")])

# Filter to existing columns
output_cols = [col for col in output_cols if col in final.columns]

final[output_cols].to_csv("predictions_all_stages_wide.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved predictions_all_stages_wide.csv")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Total predictions: {len(final_pred)}")
print(f"Countries: {df['countrynew'].nunique()}")
print(f"Stages: {len(all_stages)}")
print(f"Models: {len([m for m in MODEL_CALLERS.keys() if m in MODELS])}")
print(f"Output files:")
print(f"  - predictions_all_stages_long.csv")
print(f"  - predictions_all_stages_wide.csv")
print(f"  - predictions_S[1-8]_country.csv (per stage)")

# Display sample results
print("\nSample results:")
print(final[output_cols].head(10))

Processing Serbia (Stage 7)...
  Calling gpt... ✓ 45.7
  Calling claude... ✓ 37.5
  Calling gemini... .✓ 31.2
  Calling llama... ✓ 29.4

Processing Sierra Leone (Stage 7)...
  Calling gpt... ✓ 25.4
  Calling claude... ✓ 37.5
  Calling gemini... ✓ 12.5
  Calling llama... ✓ 23.4

Processing Singapore (Stage 7)...
  Calling gpt... ✓ 65.4
  Calling claude... ✓ 42.5
  Calling gemini... .✓ 45.0
  Calling llama... ✓ 34.6

Processing Slovakia (Stage 7)...
  Calling gpt... ✓ 35.7
  Calling claude... ✓ 32.5
  Calling gemini... ✓ 23.5
  Calling llama... ✓ 34.6

Processing Slovenia (Stage 7)...
  Calling gpt... ✓ 45.7
  Calling claude... ✓ 42.5
  Calling gemini... .✓ 27.5
  Calling llama... ✓ 34.6

Processing South Africa (Stage 7)...
  Calling gpt... ✓ 45.7
  Calling claude... ✓ 42.5
  Calling gemini... ✓ 28.5
  Calling llama... ✓ 34.5

Processing South Korea (Stage 7)...
  Calling gpt... ✓ 45.7
  Calling claude... ✓ 42.5
  Calling gemini... .✓ 30.5
  Calling llama... ✓ 24.5

Processing Spain (St

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>